# Trader Performance vs Market Sentiment - Primetrade Assignment

trying to figure out if the fear/greed index actually affects how traders behave on hyperliquid. let's see what the data says.


## setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('charts', exist_ok=True)


## loading the data first

In [ ]:
fg = pd.read_csv('fear_greed_index.csv')
tr = pd.read_excel('historical_data.xlsx')

print(fg.shape, tr.shape)


In [ ]:
# checking what columns we have
print(fg.columns.tolist())
print(tr.columns.tolist())


In [ ]:
fg.head()


In [ ]:
tr.head(3)


In [ ]:
# checking for nulls / duplicates
print("fg nulls:", fg.isnull().sum().sum(), "| dupes:", fg.duplicated().sum())
print("tr nulls:", tr.isnull().sum().sum(), "| dupes:", tr.duplicated().sum())
# clean! no missing values which is nice


## fixing timestamps and merging

In [ ]:
# timestamp in trader data is unix ms, need to convert
tr['date'] = pd.to_datetime(tr['Timestamp'], unit='ms').dt.date
tr['datetime'] = pd.to_datetime(tr['Timestamp'], unit='ms')

# check if it looks right
print(tr['date'].min(), tr['date'].max())


In [ ]:
fg['date'] = pd.to_datetime(fg['date']).dt.date

# i'll simplify the 4 categories into just Fear vs Greed for now
# (Extreme Fear = Fear, Extreme Greed = Greed)
fg['sentiment'] = fg['classification'].apply(lambda x: 'Greed' if 'Greed' in str(x) else 'Fear')
fg = fg.drop_duplicates('date')

print(fg['sentiment'].value_counts())


In [ ]:
# merge on date
tr_merged = tr.merge(fg[['date','sentiment','value','classification']], on='date', how='inner')
print("merged rows:", len(tr_merged))
print("date range:", tr_merged['date'].min(), "to", tr_merged['date'].max())


In [ ]:
# also want a separate df for just close trades (realized pnl)
closes = tr[tr['Direction'].str.contains('Close|Sell|Buy', na=False) &
            ~tr['Direction'].isin(['Spot Dust Conversion','Settlement','Auto-Deleveraging'])].copy()
cl_merged = closes.merge(fg[['date','sentiment','value','classification']], on='date', how='inner')
print("close trades:", len(cl_merged))


In [ ]:
# what are all the direction types?
tr['Direction'].value_counts()


## building the key metrics

In [ ]:
# daily pnl per trader
daily_pnl = (cl_merged.groupby(['Account','date','sentiment'])
             .agg(daily_pnl=('Closed PnL','sum'),
                  n_trades=('Trade ID','count'),
                  avg_size=('Size USD','mean'))
             .reset_index())

daily_pnl['win'] = (daily_pnl['daily_pnl'] > 0).astype(int)
daily_pnl.head()


In [ ]:
# leverage proxy - size / abs(start position)
# not perfect but gives an idea of leverage used
tr_merged['is_long'] = tr_merged['Direction'].str.contains('Long', na=False).astype(int)
tr_merged['leverage_proxy'] = np.where(
    tr_merged['Start Position'] != 0,
    tr_merged['Size USD'] / tr_merged['Start Position'].abs(),
    np.nan)

print("leverage proxy stats:")
print(tr_merged['leverage_proxy'].describe())


In [ ]:
# trader level summary
trader_stats = (daily_pnl.groupby('Account')
                .agg(total_pnl=('daily_pnl','sum'),
                     avg_daily_pnl=('daily_pnl','mean'),
                     win_rate=('win','mean'),
                     n_trading_days=('date','nunique'),
                     avg_trades_day=('n_trades','mean'))
                .reset_index())

lev = tr_merged.groupby('Account')['leverage_proxy'].median().reset_index().rename(columns={'leverage_proxy':'med_leverage'})
ls  = tr_merged.groupby('Account')['is_long'].mean().reset_index().rename(columns={'is_long':'long_ratio'})

trader_stats = trader_stats.merge(lev, on='Account').merge(ls, on='Account')
print(trader_stats.shape)
trader_stats.describe().round(2)


## analysis

### Q1: does pnl differ between fear and greed days?


In [ ]:
sent_pnl = (daily_pnl.groupby('sentiment')
            .agg(avg_pnl=('daily_pnl','mean'),
                 med_pnl=('daily_pnl','median'),
                 win_rate=('win','mean'),
                 count=('daily_pnl','count'))
            .reset_index())
print(sent_pnl.round(2))


In [ ]:
# quick t-test to see if the difference is significant
t, p = stats.ttest_ind(
    daily_pnl[daily_pnl.sentiment=='Fear']['daily_pnl'],
    daily_pnl[daily_pnl.sentiment=='Greed']['daily_pnl'])
print(f"t={t:.3f}, p={p:.4f}")
# p > 0.05 so not statistically significant
# but fear days do show higher avg pnl - probably higher variance


In [ ]:
# plotting pnl distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Performance: Fear vs Greed Days', fontsize=13, fontweight='bold')

ax = axes[0]
for s, c in [('Fear','#e05c5c'), ('Greed','#4ecb71')]:
    ax.hist(daily_pnl[daily_pnl.sentiment==s]['daily_pnl'].clip(-5000,5000),
            bins=60, alpha=0.6, color=c, label=s, density=True)
ax.set_title('PnL Distribution (clipped)')
ax.set_xlabel('Daily PnL (USD)')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
vals = [sent_pnl[sent_pnl.sentiment==s]['avg_pnl'].values[0] for s in ['Fear','Greed']]
ax.bar(['Fear','Greed'], vals, color=['#e05c5c','#4ecb71'], width=0.5, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title('Avg Daily PnL')
ax.set_ylabel('USD')
ax.grid(alpha=0.3, axis='y')
for i, v in enumerate(vals):
    ax.text(i, v + 1000, f'${v:,.0f}', ha='center', fontsize=9, fontweight='bold')

ax = axes[2]
wr = [sent_pnl[sent_pnl.sentiment==s]['win_rate'].values[0]*100 for s in ['Fear','Greed']]
ax.bar(['Fear','Greed'], wr, color=['#e05c5c','#4ecb71'], width=0.5, edgecolor='black', linewidth=0.5)
ax.set_title('Win Rate (%)')
ax.set_ylabel('%')
ax.set_ylim(0, 100)
ax.grid(alpha=0.3, axis='y')
for i, v in enumerate(wr):
    ax.text(i, v+1, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('charts/chart1_pnl_performance.png', dpi=150, bbox_inches='tight')
plt.show()


### Q2: do traders behave differently on fear vs greed days?

In [ ]:
# aggregate daily market behavior
daily_market = (tr_merged.groupby(['date','sentiment'])
                .agg(n_trades=('Trade ID','count'),
                     avg_size=('Size USD','mean'),
                     long_ratio=('is_long','mean'),
                     avg_lev=('leverage_proxy','median'))
                .reset_index())

behavior = daily_market.groupby('sentiment').agg(
    avg_trades=('n_trades','mean'),
    avg_size=('avg_size','mean'),
    long_ratio=('long_ratio','mean'),
    avg_lev=('avg_lev','mean')).reset_index()

print(behavior.round(3))


In [ ]:
# interesting - greed days have way higher leverage proxy (probably outliers)
# and more long bias which makes sense - people buy when greedy

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Trader Behavior by Sentiment', fontsize=13, fontweight='bold')

metrics = [
    ('avg_trades', 'Avg Trades/Day', 'count'),
    ('avg_size',   'Avg Trade Size', 'USD'),
    ('long_ratio', 'Long Ratio',     '0-1'),
    ('avg_lev',    'Median Leverage Proxy', 'x'),
]
colors = ['#e05c5c','#4ecb71']
for ax, (col, title, unit) in zip(axes.flat, metrics):
    vals = [behavior[behavior.sentiment==s][col].values[0] for s in ['Fear','Greed']]
    ax.bar(['Fear','Greed'], vals, color=colors, width=0.5, edgecolor='black', linewidth=0.5)
    ax.set_title(title); ax.set_ylabel(unit); ax.grid(alpha=0.3, axis='y')
    for i, v in enumerate(vals):
        ax.text(i, v*1.01, f'{v:.2f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('charts/chart2_behavior.png', dpi=150, bbox_inches='tight')
plt.show()


### Q3: trader segments

In [ ]:
# split traders into segments based on their overall stats
# using median as cutoff (simple but effective for 32 traders)

lev_med  = trader_stats['med_leverage'].median()
freq_med = trader_stats['n_trading_days'].median()

trader_stats['lev_segment']  = np.where(trader_stats['med_leverage']   > lev_med,  'High Leverage', 'Low Leverage')
trader_stats['freq_segment'] = np.where(trader_stats['n_trading_days'] > freq_med, 'Frequent', 'Infrequent')
trader_stats['perf_segment'] = pd.cut(trader_stats['win_rate'],
                                       bins=[0,.4,.6,1.],
                                       labels=['Inconsistent','Moderate','Consistent Winner'])

print(trader_stats['lev_segment'].value_counts().to_dict())
print(trader_stats['freq_segment'].value_counts().to_dict())
print(trader_stats['perf_segment'].value_counts().to_dict())


In [ ]:
# merge segments back and check pnl by segment x sentiment
daily_seg = daily_pnl.merge(
    trader_stats[['Account','lev_segment','freq_segment','perf_segment']], on='Account', how='left')

# helper to get avg pnl by segment x sentiment
def seg_summary(df, seg_col):
    return df.groupby([seg_col,'sentiment']).agg(
        avg_pnl=('daily_pnl','mean'), win_rate=('win','mean')).reset_index()

lev_perf  = seg_summary(daily_seg, 'lev_segment')
freq_perf = seg_summary(daily_seg, 'freq_segment')
perf_perf = seg_summary(daily_seg, 'perf_segment')

print("lev segment:")
print(lev_perf.round(1))


In [ ]:
# freq segment is interesting
print("freq segment:")
print(freq_perf.round(1))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Trader Segments x Sentiment', fontsize=13, fontweight='bold')

def seg_bar(ax, df, seg_col, title):
    segs = df[seg_col].dropna().unique()
    x = np.arange(len(segs)); w = 0.35
    fv = [df[(df[seg_col]==s)&(df.sentiment=='Fear')]['daily_pnl'].mean() for s in segs]
    gv = [df[(df[seg_col]==s)&(df.sentiment=='Greed')]['daily_pnl'].mean() for s in segs]
    ax.bar(x-w/2, fv, w, color='#e05c5c', label='Fear',  edgecolor='black', linewidth=0.5)
    ax.bar(x+w/2, gv, w, color='#4ecb71', label='Greed', edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_xticks(x); ax.set_xticklabels([str(s) for s in segs], fontsize=8)
    ax.set_ylabel('Avg Daily PnL'); ax.set_title(title); ax.legend(); ax.grid(alpha=0.3, axis='y')

seg_bar(axes[0], daily_seg, 'lev_segment',  'High vs Low Leverage')
seg_bar(axes[1], daily_seg, 'freq_segment', 'Frequent vs Infrequent')
seg_bar(axes[2], daily_seg, 'perf_segment', 'Performance Tier')

plt.tight_layout()
plt.savefig('charts/chart3_segments.png', dpi=150, bbox_inches='tight')
plt.show()


### timeline - pnl vs fg index over time

In [ ]:
daily_agg = (cl_merged.groupby('date')
             .agg(total_pnl=('Closed PnL','sum'), n_trades=('Trade ID','count'))
             .reset_index())
daily_agg = daily_agg.merge(fg[['date','value','sentiment']], on='date', how='left')
daily_agg['date_dt'] = pd.to_datetime(daily_agg['date'])
daily_agg = daily_agg.sort_values('date_dt')
daily_agg['pnl_7d'] = daily_agg['total_pnl'].rolling(7).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('PnL Timeline vs Fear/Greed Index', fontsize=13, fontweight='bold')

# shade background by sentiment
for ax in [ax1, ax2]:
    for _, row in daily_agg.iterrows():
        c = '#4ecb71' if row['sentiment']=='Greed' else '#e05c5c'
        ax.axvspan(row['date_dt'], row['date_dt']+pd.Timedelta(days=1), color=c, alpha=0.07)

ax1.plot(daily_agg['date_dt'], daily_agg['pnl_7d'], color='steelblue', lw=2, label='7d rolling PnL')
ax1.axhline(0, color='black', lw=0.6, ls='--')
ax1.set_ylabel('Total PnL (USD)'); ax1.set_title('Aggregate PnL (7d MA)')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(daily_agg['date_dt'], daily_agg['value'], color='orange', lw=1.5)
ax2.fill_between(daily_agg['date_dt'], daily_agg['value'], 50,
                 where=daily_agg['value']>=50, color='#4ecb71', alpha=0.3, label='Greed')
ax2.fill_between(daily_agg['date_dt'], daily_agg['value'], 50,
                 where=daily_agg['value']<50,  color='#e05c5c', alpha=0.3, label='Fear')
ax2.axhline(50, color='black', lw=0.6, ls='--')
ax2.set_ylabel('FG Index'); ax2.set_title('Fear & Greed Index')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('charts/chart4_timeline.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Leverage & Positioning by Sentiment', fontsize=13, fontweight='bold')

ax = axes[0]
for s, c in [('Fear','#e05c5c'),('Greed','#4ecb71')]:
    d = tr_merged[tr_merged.sentiment==s]['leverage_proxy'].dropna()
    d = d[(d>0)&(d<50)]
    ax.hist(d, bins=50, alpha=0.6, color=c, label=s, density=True)
ax.set_xlabel('Leverage Proxy'); ax.set_ylabel('Density')
ax.set_title('Leverage Distribution'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ls_sent = (tr_merged[tr_merged.Direction.isin(['Open Long','Open Short'])]
           .groupby('sentiment')['is_long'].mean().reset_index())
ls_sent['short_ratio'] = 1 - ls_sent['is_long']
x = np.arange(len(ls_sent)); w=0.35
ax.bar(x-w/2, ls_sent['is_long']*100,     w, color='#4ecb71', label='Long %',  edgecolor='black', linewidth=0.5)
ax.bar(x+w/2, ls_sent['short_ratio']*100, w, color='#e05c5c', label='Short %', edgecolor='black', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(ls_sent['sentiment'])
ax.set_ylabel('%'); ax.set_title('Long/Short Ratio by Sentiment')
ax.legend(); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('charts/chart5_leverage_ls.png', dpi=150, bbox_inches='tight')
plt.show()


## findings & strategy ideas

### what i found:

**1. fear days = higher pnl, but not consistently**
- avg pnl on fear days: ~$169k vs ~$96k on greed days
- win rate also slightly higher (85% vs 83%)
- BUT p-value = 0.32 so not statistically significant with this sample
- probably because fewer traders are active on fear days, and the ones who do trade are more careful

**2. infrequent traders are really good at picking fear day entries**
- infrequent traders: $234k avg on fear days vs $62k on greed days (3.75x difference!)
- they basically wait for market fear to make big moves

**3. high leverage on greed days = bad idea**
- high-lev traders drop from $173k on fear to only $32k on greed (~81% drop)
- greed markets are choppy and mean-reverting - leverage gets you wrecked

**4. consistent winners don't care about sentiment**  
- they make ~$130k on both fear and greed days
- suggests they have a real edge, not just betting on sentiment direction

**5. long bias spikes on greed days (64% vs 40%)**
- everyone piling into longs when market is greedy = overcrowded trade

---

### strategy recommendations:

**Strategy 1 - "Fear-First" for leveraged traders**
- if you use high leverage: be MORE active on fear days, pull back on greed days
- reduce position sizes by ~50% when FG index > 60

**Strategy 2 - "Wait & Strike" for patient traders**  
- if you trade infrequently: focus your best setups on fear periods (FG < 40)
- during greed, hold winners rather than opening new positions


In [ ]:
print("done! charts saved to ./charts/")
